# step1 — 준수율 절벽 (RQ1)

**무엇을 확인하나:** 문맥(함수 12개)에 **규약 위반 이름(snake)이 많아질수록**, 모델이 새로 쓰는 함수 이름의
**준수율이 떨어지는가**, 어디서 급락(절벽)하는가. 실패가 실재함을 보이는 스텝(뒤 스텝들의 전제).

**어떻게:** 지침은 'camelCase 써라'로 고정. 문맥의 **위반 개수를 0~12 전부** 바꿔가며, 모델이 실제로
**생성한 첫 함수 이름**을 뽑아 camel인지 본다. **두 언어(파이썬·자바스크립트)** 로 각각.

**고정 설정:** 4모델 · 504이름(42묶음) · 무작위값 42 · 그리디 생성 1턴.

> **언어 관용 주의:** 파이썬 관용=snake, JS 관용=camel. 그래서 'camelCase 써라'는 파이썬엔 관용과 반대,
> JS엔 일치 → **언어별 기본 준수율이 다름**(결함 아니라 대조점). 절벽이 두 언어 다 나오나를 본다.

**모델 하나씩.** 셀 ③ `PICK` → 셀 ④~⑦. 끝나면 `PICK` 바꿔 반복. 끊겨도 저장된 건 건너뜀.


In [ ]:
# ① 환경 설정 — 설치, GPU 확인, 무작위값 42 고정
!pip install -q transformers accelerate torch matplotlib pandas bitsandbytes

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (매우 느림)')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('무작위값 고정:', SEED)


In [ ]:
# ② 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step1/cliff
!git checkout step1/cliff
!git pull --quiet origin step1/cliff
!pip install -e . -q
import sys; sys.path.insert(0, 'src')


In [ ]:
# ③ 조건 설정 — 4모델 중 하나 골라 위반개수 0~12 x 2언어 x 42묶음
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation)

MODELS = [
    ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct',           family='qwen',      dtype='float16'),
    ModelSpec(name='deepseek-ai/deepseek-coder-6.7b-instruct',  family='deepseek',  dtype='float16'),
    ModelSpec(name='unsloth/Llama-3.2-3B-Instruct',             family='llama',     dtype='float16'),
    ModelSpec(name='stabilityai/stable-code-instruct-3b',       family='stability', dtype='float16'),
]

# ★ 이번에 돌릴 모델 하나 (0=qwen, 1=deepseek, 2=llama, 3=stable)
PICK = 0
MODEL = MODELS[PICK]
# deepseek-6.7b OOM나면 아래 해제(8bit):
# if MODEL.family=='deepseek': MODEL = ModelSpec(name=MODEL.name, family='deepseek', dtype='float16', quantization='8bit')

LANGS  = [None, 'javascript']       # None=파이썬
NCOMP  = list(range(13))            # 준수 개수 0~12 (= 위반 12~0)
BLOCKS = list(range(42))

def cond(lang, ncomp, block):
    return Condition(model=MODEL,
        preceding=PrecedingCode(n_compliant=ncomp, n_functions=12, composition=Composition.POOL,
                                pool_block=block, lang=lang),
        instruction=Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL),
        seed=42)

conditions = [cond(l, nc, b) for l in LANGS for nc in NCOMP for b in BLOCKS]
print('모델:', MODEL.family, '| 조건 수:', len(conditions), '(=2언어 x 13점 x 42묶음)')
print('예(파이썬 위반6):', cond(None, 6, 0).slug())
print('예(JS 위반6):', cond('javascript', 6, 0).slug())


In [ ]:
# ④ 실행 — 생성(그리디 1턴). 조건마다 즉시 저장(재개). 첫 함수 이름 준수 여부 측정.
from harness import run, ResultRecord, save_result, result_path
from harness.model import load_model

STEP = 'step1_cliff'
todo = [c for c in conditions if not result_path(c, step=STEP).exists()]
print(f'[{MODEL.family}] 전체 {len(conditions)} / 남은 {len(todo)}')

if todo:
    handle = load_model(MODEL)
    print(f'  층수 {handle.num_layers} | 로드 완료')
    for i, c in enumerate(todo, 1):
        out = run(c, handle=handle, max_turns=1, max_new_tokens=100)   # 첫 함수만 짧게 생성
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics, step=STEP, rq='RQ1'))
        if i % 50 == 0 or i == len(todo):
            ex = out.metrics.extra
            lang = c.preceding.lang or 'python'
            nviol = c.preceding.n_functions - c.preceding.n_compliant
            nm = ex['turn_names'][0]; ok = '준수' if ex['first_compliant'] else '위반'
            print(f"    [{i}/{len(todo)}] {lang} 위반{nviol}개 -> 생성 '{nm}' ({ok})")
    print('  완료.')
else:
    print('  이미 다 됨')


In [ ]:
# ⑤ 결과 로드 (이 모델 것)
from harness import result_path
from harness.results import load_result
recs = [load_result(result_path(c, step='step1_cliff')) for c in conditions
        if result_path(c, step='step1_cliff').exists()]
print('불러온 조건:', len(recs), '-> results/step1_cliff/')


In [ ]:
# ⑥ 요약 — 위반개수 vs 준수율 (언어별, 절벽 곡선) + 신뢰구간 + 생성 예시 + 그림
import numpy as np, pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt

# (언어, 위반개수) -> [준수 0/1] , 이름 못 뽑은 비율도
agg = defaultdict(lambda: defaultdict(list))
parsefail = defaultdict(lambda: defaultdict(int))
for r in recs:
    lang = r.condition.preceding.lang or 'python'
    nviol = r.condition.preceding.n_functions - r.condition.preceding.n_compliant
    ex = r.metrics.extra
    agg[lang][nviol].append(1 if ex['first_compliant'] else 0)
    if ex['turn_names'][0] is None: parsefail[lang][nviol] += 1

def rate_ci(vals):
    a = np.array(vals); p = a.mean(); n = len(a)
    ci = 1.96*np.sqrt(p*(1-p)/n) if n>0 else 0
    return p, ci

print('=== 언어별 위반개수 vs 준수율 ===')
for lang in ['python','javascript']:
    if lang not in agg: continue
    xs = sorted(agg[lang]); ps=[]; 
    for x in xs:
        p,ci = rate_ci(agg[lang][x]); ps.append(p)
        print(f'  {lang:10s} 위반 {x:2d}개: 준수율 {p*100:5.1f}% (±{ci*100:.1f})  이름못뽑음 {parsefail[lang][x]}')
    print(f'    -> {lang}: 위반0 {ps[0]*100:.0f}% ... 위반12 {ps[-1]*100:.0f}%')

# 절벽 그림 (영문 라벨)
fig, ax = plt.subplots(figsize=(7.5,4.5))
for lang,color in [('python','#2563eb'),('javascript','#ea580c')]:
    if lang not in agg: continue
    xs = sorted(agg[lang]); P=[]; C=[]
    for x in xs:
        p,ci = rate_ci(agg[lang][x]); P.append(p); C.append(ci)
    P=np.array(P); C=np.array(C)
    ax.plot(xs, P, marker='o', color=color, lw=2, label=lang)
    ax.fill_between(xs, P-C, P+C, color=color, alpha=.18)
ax.set_xlabel('Number of violation names in context (0-12)')
ax.set_ylabel('Compliance rate (first generated name)')
ax.set_title(f'{MODEL.family} — compliance cliff (step1, RQ1)')
ax.set_ylim(-0.02,1.02); ax.grid(alpha=.25); ax.legend(); plt.tight_layout(); plt.show()

# 생성 예시 몇 개 (눈으로 검증)
print('\n=== 생성 예시 (위반12개, 파이썬) ===')
shown=0
for r in recs:
    if shown>=3: break
    if (r.condition.preceding.lang or 'python')=='python' and r.condition.preceding.n_compliant==0:
        t=r.metrics.extra['turn_texts'][0].replace(chr(10),' ')[:90]
        print(f"  이름 '{r.metrics.extra['turn_names'][0]}' ({r.metrics.extra['turn_notations'][0]}): {t}")
        shown+=1


In [ ]:
# ⑦ 결과 폴더 zip 다운로드
import shutil
shutil.make_archive('step1_cliff_results', 'zip', 'results/step1_cliff')
try:
    from google.colab import files; files.download('step1_cliff_results.zip')
except Exception as e:
    print('Colab 아님(로컬):', e)
